In [2]:
!pip install -q -U google-genai

In [3]:
from google import genai
from google.colab import userdata
from google.genai import types
import json
from pydantic import BaseModel #base model for Schema defn
from typing import Optional  #handling null values

client=genai.Client(api_key=userdata.get("GEMINI_API_KEYS"))
MODEL="gemini-3.5-flash-lite"

In [10]:
user_input=input("Enter your information,tech stack")
prompt=f"""
Act as senior HR,
consider the Student information provided in {user_input} and then return Structured data in JSON format.
JSON format must contain keys(name(string),dob(string),education(string),skills(list),projects(list)) , if student information does not provide values for any key then make that key as null.
Dont add any garbage or assumed value to any key.
for dob return string in standard indian format(dd month(in letters) year)
"""
class Resume(BaseModel):
  name:Optional[str]
  dob:Optional[str]
  education:Optional[str]
  skills:Optional[list]
  projects:Optional[list]

response=client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config=types.GenerateContentConfig(
        temperature=0.4,
        max_output_tokens=1500,
        system_instruction="You are resume parser, reply every time with (Hi I am your resume assitant,Provide me with your information i will format) ",
        response_mime_type="application/json",
        response_schema=Resume,
        thinking_config=types.ThinkingConfig(thinking_level='low')
    )
)
data_1=json.loads(response.text)
for key in data_1:
  data_1.setdefault(key,None)
print(json.dumps(data_1,indent=2))

Enter your information,tech stackMy name is kushal,i am studying 3rd year engineering in CSE. I learnt python,c,c++,webdevelopment and i built smart health monitoring ear tags and erp project . I born on 12 10 2006
{
  "name": "Kushal",
  "dob": "12 October 2006",
  "education": "3rd year engineering in CSE",
  "skills": [
    "python",
    "c",
    "c++",
    "webdevelopment"
  ],
  "projects": [
    "smart health monitoring ear tags",
    "erp project"
  ]
}
